In [ ]:
import glob
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def parse_ninja_file(filepath, is_pv=True):
    valid_rows = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if len(line) >= 10 and line[:4].isdigit() and line[4] == "-":
                parts = [p.strip() for p in line.strip().split(",")]
                if len(parts) >= 5:
                    valid_rows.append(parts[:5])

    cols = (
        ["time", "electricity", "irradiance_direct", "irradiance_diffuse", "temp"]
        if is_pv
        else ["time", "precsnoland", "snomas", "swtdn", "cldtot"]
    )
    df = pd.DataFrame(valid_rows, columns=cols)

    for col in cols[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["time"] = pd.to_datetime(df["time"])

    keep_cols = ["time", "electricity"] if is_pv else ["time", "cldtot"]
    return (
        df[keep_cols]
        .drop_duplicates(subset=["time"])
        .sort_values("time")
        .reset_index(drop=True)
    )

def parse_smhi_multi(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    header_idx = next(
        i
        for i, line in enumerate(lines)
        if line.startswith("Datum;") or line.startswith("Date;")
    )
    header_cols = [c.strip() for c in lines[header_idx].strip().split(";")]

    valid_rows = []
    for line in lines[header_idx + 1 :]:
        parts = [p.strip() for p in line.strip().split(";")]
        if (
            len(parts) >= 3
            and len(parts[0]) == 10
            and parts[0][:4].isdigit()
            and parts[0][4] == "-"
            and parts[0][7] == "-"
        ):
            if len(parts[1]) >= 5 and parts[1][2] == ":":
                valid_rows.append(parts)

    if not valid_rows:
        return pd.DataFrame(columns=["time"])

    max_len = max(len(r) for r in valid_rows)
    cols = (
        header_cols + [f"extra_{i}" for i in range(max_len - len(header_cols))]
        if max_len > len(header_cols)
        else header_cols[:max_len]
    )

    padded_rows = [r + [""] * (len(cols) - len(r)) for r in valid_rows]
    df = pd.DataFrame(padded_rows, columns=cols)

    date_col = "Datum" if "Datum" in df.columns else "Date"
    time_col = "Tid (UTC)" if "Tid (UTC)" in df.columns else "Time"

    df["time"] = pd.to_datetime(df[date_col] + " " + df[time_col], errors="coerce")
    df = df.dropna(subset=["time"])

    ignore_cols = [date_col, time_col, "Kvalitet", "Quality", "", "Tidsutsnitt:"] + [
        c for c in df.columns if c.startswith("extra_")
    ]
    value_cols = [c for c in df.columns if c not in ignore_cols and c != "time"]

    for vc in value_cols:
        df[vc] = pd.to_numeric(df[vc], errors="coerce")

    return (
        df[["time"] + value_cols]
        .drop_duplicates("time")
        .sort_values("time")
        .reset_index(drop=True)
    )


pv_files = sorted(glob.glob("ninja_pv_*.csv"))
df_pv = pd.concat([parse_ninja_file(f, is_pv=True) for f in pv_files], ignore_index=True)

weather_files = sorted(glob.glob("ninja_20*.csv"))
weather_files = [f for f in weather_files if "ninja_pv" not in f]
df_cloud = pd.concat(
    [parse_ninja_file(f, is_pv=False) for f in weather_files], ignore_index=True
)

df_final = df_pv.merge(df_cloud, on="time", how="left")

smhi_files = [
    "air_temperature_2020_2025.csv",
    "wind_2020_2025.csv",
    "relative_humidity_2020_2025.csv",
    "pressure_2020_2025.csv",
    "irradiance_sunshinetime_2020_2025.csv",
]

for file_path in smhi_files:
    if glob.glob(file_path):
        df_smhi_var = parse_smhi_multi(file_path)
        df_final = df_final.merge(df_smhi_var, on="time", how="left")

df_final = (
    df_final.sort_values("time")
    .interpolate(method="linear", limit=3)
    .bfill()
    .ffill()
    .reset_index(drop=True)
)

df_final["electricity"] = df_final["electricity"].clip(lower=0.0)

df_final["hour_sin"] = np.sin(2 * np.pi * df_final["time"].dt.hour / 24.0)
df_final["hour_cos"] = np.cos(2 * np.pi * df_final["time"].dt.hour / 24.0)
df_final["doy_sin"] = np.sin(2 * np.pi * df_final["time"].dt.dayofyear / 365.25)
df_final["doy_cos"] = np.cos(2 * np.pi * df_final["time"].dt.dayofyear / 365.25)

feature_candidates = [
    "Lufttemperatur",
    "Vindhastighet",
    "Vindriktning",
    "Relativ Luftfuktighet",
    "Lufttryck reducerat havsytans nível",
    "Solskenstid",
    "cldtot",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
]

feature_cols = [col for col in feature_candidates if col in df_final.columns]

X = df_final[feature_cols].values
y = df_final["electricity"].values
y_binary = (y > 0.001).astype(int)

train_mask = (df_final["time"].dt.year < 2025).values
test_mask = (df_final["time"].dt.year == 2025).values

X_train_raw, X_test_raw = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
y_train_b, y_test_b = y_binary[train_mask], y_binary[test_mask]

In [ ]:
def calc_regression_metrics(y_true, y_pred_raw):
    mae = mean_absolute_error(y_true, y_pred_raw)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_raw))
    
    
    if np.var(y_true) == 0:
        r2_str = "N/A (Var=0)"
    else:
        r2_str = f"{r2_score(y_true, y_pred_raw):.4f}"

    return {
        "MAE (kW)": f"{mae:.4f}",
        "RMSE (kW)": f"{rmse:.4f}",
        "R² Score": r2_str
    }

def calc_classification_metrics(y_true_b, y_pred_b):
    acc = accuracy_score(y_true_b, y_pred_b)
    prec = precision_score(y_true_b, y_pred_b, zero_division=0)
    rec = recall_score(y_true_b, y_pred_b, zero_division=0)
    f1 = f1_score(y_true_b, y_pred_b, zero_division=0)
    
    return {
        "Accuracy": f"{acc:.4f}",
        "Precision": f"{prec:.4f}",
        "Recall": f"{rec:.4f}",
        "F1-Score": f"{f1:.4f}"
    }

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)


print("LogR")
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

lr_grid = GridSearchCV(
    pipe_lr,
    param_grid={"model__C": [0.1, 1.0, 10.0]},
    cv=tscv,
    scoring="f1",
    n_jobs=-1,
).fit(X_train_raw, y_train_b)


print("LinR")
pipe_lin = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
]).fit(X_train_raw, y_train)


print("SNN")
pipe_snn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(max_iter=500, early_stopping=False, random_state=42))
])

snn_grid = GridSearchCV(
    pipe_snn,
    param_grid={
        "model__hidden_layer_sizes": [(32,), (64,)],
        "model__alpha": [0.0001, 0.001],
        "model__activation": ["relu", "tanh"],
        "model__learning_rate_init": [0.001, 0.01],
    },
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train_raw, y_train)


print("DNN")
pipe_dnn = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(max_iter=500, early_stopping=False, random_state=42))
])

dnn_grid = GridSearchCV(
    pipe_dnn,
    param_grid={
        "model__hidden_layer_sizes": [(64, 32), (128, 64, 32)],
        "model__alpha": [0.0001, 0.001],
        "model__activation": ["relu", "tanh"],
        "model__learning_rate_init": [0.001, 0.01],
    },
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train_raw, y_train)

print("\nTRAINING DONE")

In [ ]:

y_train_pred_lr = lr_grid.predict(X_train_raw)
y_test_pred_lr = lr_grid.predict(X_test_raw)

y_train_pred_lin_raw = pipe_lin.predict(X_train_raw)
y_test_pred_lin_raw = pipe_lin.predict(X_test_raw)

y_train_pred_snn_raw = snn_grid.predict(X_train_raw)
y_test_pred_snn_raw = snn_grid.predict(X_test_raw)

y_train_pred_dnn_raw = dnn_grid.predict(X_train_raw)
y_test_pred_dnn_raw = dnn_grid.predict(X_test_raw)


reg_models = {
    "Linear Regression": (y_train_pred_lin_raw, y_test_pred_lin_raw),
    "Shallow NN": (y_train_pred_snn_raw, y_test_pred_snn_raw),
    "Deep NN": (y_train_pred_dnn_raw, y_test_pred_dnn_raw),
}


lr_train = calc_classification_metrics(y_train_b, y_train_pred_lr)
lr_test = calc_classification_metrics(y_test_b, y_test_pred_lr)
df_lr = pd.DataFrame([lr_train, lr_test], index=["Train (2020-2024)", "Test (2025)"])

print("==================================================")
print(" 1. LOGISTIC REGRESSION")
print("==================================================")
print(df_lr.T.to_string())


print("\n==================================================")
print(" 2. REGRESSION MODELS")
print("==================================================")

for name, (pred_train, pred_test) in reg_models.items():
    m_train = calc_regression_metrics(y_train, pred_train)
    m_test = calc_regression_metrics(y_test, pred_test)
    
    df_reg = pd.DataFrame([m_train, m_test], index=["Train (2020-2024)", "Test (2025)"])
    print(f"\n--- {name.upper()} ---")
    print(df_reg.T.to_string())

In [ ]:
diag_rows = []

for name, (_, y_raw_test) in reg_models.items():
    mae_raw = mean_absolute_error(y_test, y_raw_test)
    rmse_raw = np.sqrt(mean_squared_error(y_test, y_raw_test))
    r2_raw = r2_score(y_test, y_raw_test)
    
    num_neg = np.sum(y_raw_test < 0)
    pct_neg = (num_neg / len(y_test)) * 100
    min_val = np.min(y_raw_test)
    
    diag_rows.append({
        "Model": name,
        "MAE(kW)": f"{mae_raw:.4f}",
        "RMSE(kW)": f"{rmse_raw:.4f}",
        "R²": f"{r2_raw:.4f}",
        "Hours<0 kW": f"{num_neg} ({pct_neg:.1f}%)",
        "Minimum(kW)": f"{min_val:.4f}",
    })

df_diag = pd.DataFrame(diag_rows)

print("==================================================")
print("   DIAGNOSIS W/OUT CLIP")
print("==================================================")
print(df_diag.to_string(index=False))

In [ ]:
train_day_mask = y_train > 0.001
train_night_mask = y_train <= 0.001

test_day_mask = y_test > 0.001
test_night_mask = y_test <= 0.001

print("==================================================")
print(" 3. ANALYSIS")
print("==================================================")
print(f"Daytime  -> Train: {np.sum(train_day_mask)} h | Test: {np.sum(test_day_mask)} h")
print(f"Nightime -> Train: {np.sum(train_night_mask)} h | Test: {np.sum(test_night_mask)} h\n")

for name, (pred_train, pred_test) in reg_models.items():
    m_train_day = calc_regression_metrics(y_train[train_day_mask], pred_train[train_day_mask])
    m_test_day = calc_regression_metrics(y_test[test_day_mask], pred_test[test_day_mask])
    
    #m_train_night = calc_regression_metrics(y_train[train_night_mask], pred_train[train_night_mask])
    #m_test_night = calc_regression_metrics(y_test[test_night_mask], pred_test[test_night_mask])
    
    df_segmented = pd.DataFrame({
        "Day (Train)": m_train_day,
        "Day (Test 2025)": m_test_day
        #"Noite (Treino)": m_train_night,
        #"Noite (Teste 2025)": m_test_night
    })
    
    print(f"--------------------------------------------------")
    print(f" RESULTS: {name.upper()}")
    print(f"--------------------------------------------------")
    print(df_segmented.to_string())
    print("\n")